In [1]:
!pip install -q faiss-cpu sentence-transformers pypdf groq



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 4.9 MB/s eta 0:00:00


In [2]:
from google.colab import files
uploaded = files.upload()


Saving Annual-Report-FY-2023-24.pdf to Annual-Report-FY-2023-24.pdf


In [3]:
import os
print(os.listdir())


['.config', 'Annual-Report-FY-2023-24.pdf', 'sample_data']


In [4]:
from pypdf import PdfReader

reader = PdfReader("Annual-Report-FY-2023-24.pdf")

text = ""
for page in reader.pages:
    text += page.extract_text() + "\n"


In [5]:
def chunk_text(text, chunk_size=800, overlap=150):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks

chunks = chunk_text(text)
print("Total Chunks:", len(chunks))


Total Chunks: 81


In [7]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(chunks, show_progress_bar=True)
embeddings = np.array(embeddings)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

In [8]:
import faiss

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("FAISS index created")


FAISS index created


In [39]:
def rag_answer(query, top_k=5):
    # Encode query
    query_vector = model.encode([query])

    # Search FAISS
    distances, indices = index.search(np.array(query_vector), top_k)

    # Get relevant chunks
    retrieved_chunks = [chunks[i] for i in indices[0]]

    context = "\n\n".join(retrieved_chunks)

    prompt = f"""
You are a strict document-grounded financial analysis assistant.

You MUST answer the question using ONLY the provided context
from the Swiggy Annual Report FY 2023–24.

STRICT RULES (MANDATORY):

1. Use ONLY information explicitly written in the context.
2. Do NOT use prior knowledge.
3. Do NOT infer, estimate, or calculate unless explicitly stated.
4. If financial tables contain multiple years, select ONLY the value
   corresponding exactly to the year mentioned in the question.
5. Copy numerical values EXACTLY as written (including ₹, commas, decimals).
6. If multiple companies are mentioned in biographies, only extract roles
   related to Swiggy (the reporting company).
7. If the information is not explicitly present, respond EXACTLY with:

   I could not find this information in the Swiggy Annual Report.

8. Do not explain your reasoning.
9. Provide a concise, factual answer only.

----------------------------------------
CONTEXT:
{context}
----------------------------------------

QUESTION:
{query}

FINAL ANSWER:
"""







    response = client.chat.completions.create(
      model="llama-3.1-8b-instant",
      messages=[{"role": "user", "content": prompt}],
      temperature=0
    )


    return response.choices[0].message.content, retrieved_chunks


In [42]:
answer, sources = rag_answer("What is the total revenue of FY 2023-24")

print("Answer:\n")
print(answer)

print("\nSources:\n")
for s in sources:
    print(s[:300])
    print("\n---\n")


Answer:

Net Sales /Income from Business Operations  112,474

Sources:

 82,646 
Other Income  3,870 4,499 
 
 
4  
Total Income 116,343 87,145 
Less: Total expenses including Depreciation  139,474 128,844 
Profit/(Loss) after depreciation and other 
expenses 
(23,130) (41,699) 
Less: Exceptional Items + Taxes (306) (93) 
Less: Share in net loss of an associate  (66) (1

---

 including Depreciation  88,020 88,860 
Profit/(Loss) after depreciation and other 
expenses 
(17,854) (35,247) 
Less: Exceptional Items + Taxes  (1026) (2,329) 
Net Profit/(Loss) after Tax  (18,880) (37,576) 
Other comprehensive income  936 (139) 
Total comprehensive loss for the year, net of tax  

---

03-08-2018 
3.  Reporting period for the subsidiary 
concerned, if different from the holding 
company’s reporting period  
NA (same as Holding Company's reporting 
period) 
4.  Reporting currency and Exchange rate as on 
the last date of the relevant financial year in 
the case of foreign subsidiar

---

ture in

In [12]:
!pip install gradio


In [43]:
import gradio as gr

# Make sure these exist
print("Index:", type(index))
print("Chunks:", len(chunks))

def ask_question(query):
    try:
        if not query.strip():
            return "Please enter a question.", ""

        # Call RAG
        answer, sources = rag_answer(query)

        source_text = "\n\n---\n\n".join([s[:500] for s in sources])

        return answer, source_text

    except Exception as e:
        import traceback
        return f"Error:\n{traceback.format_exc()}", ""

with gr.Blocks() as demo:
    gr.Markdown("#  Financial RAG Assistant")

    query_input = gr.Textbox(label="Enter your question")
    answer_output = gr.Textbox(label="Answer", lines=6)
    source_output = gr.Textbox(label="Retrieved Sources", lines=10)

    ask_button = gr.Button("Ask")

    ask_button.click(
        ask_question,
        inputs=query_input,
        outputs=[answer_output, source_output]
    )

demo.launch(share=True)


Index: <class 'faiss.swigfaiss_avx2.IndexFlatL2'>
Chunks: 81
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d5a850714d69256b82.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
